# European Option - Binomial Tree Model

In [1]:
import math
import numpy as np
from typing import Union, NamedTuple, Literal
import scipy

In [2]:
def EuroPricerBinomial(S: float, K: float, sigma: float, r: float, q: float, T: float, N: float,
                 option_type: Literal["call", "put"] = "call"
                 ):
  """
  European option pricer using a binomial step model.

  Args:
    S: Spot price
    K: Strike price
    sigma: Volatility (annualized)
    r: Risk-free rate
    q: Dividend yield
    T: Time to maturity (years)
    N: Number of time steps
    option_type: "call" or "put"

  Returns:
    EurPricerResult: price of the option
  """
  # Time per step
  dT = T / N

  # up/down factors
  u = np.exp(sigma * np.sqrt(dT))
  d = 1 / u

  # Risk-neutral probability
  p = (np.exp((r - q) * dT) - d) / (u - d)
  # --- Input validation ---
  if not (0 < p < 1):
    raise ValueError("Risk-neutral probability outside [0,1]. Check parameters.")

  # Discount factor
  discFact = np.exp(-r * dT)

  # --- Step 1: Stock prices at maturity ---
  j = np.arange(N + 1)
  S_T = S * (u ** j) * (d ** (N - j))

  # --- Step 2: Payoff at maturity (intrinsic value) ---
  if option_type == "call":
    option_value = np.maximum(S_T - K, 0)
  else:
    option_value = np.maximum(K - S_T, 0)

  # --- Step 3: Backward induction (NO EARLY EXERCISE) ---
  for i in range(N - 1, -1, -1):
    # Continuation value (discounted expected value of holding)
    # For European options, this is the ONLY value at each node
    option_value = discFact * (p * option_value[1:i+2] + (1 - p) * option_value[:i+1])

  return option_value[0]

# Example usage
eur_call = EuroPricerBinomial(S=100, K=105, sigma=0.2, r=0.05, q=0.0, T=5, N=200, option_type="call")
print(f"European Call: {eur_call:.4f}")

European Call: 26.7622
